# 数据下载脚本

本文件有三个用途：

1. **生成讲义数据**：下载 `data_store_manage.ipynb` 所需的全部原始数据，存入 `data_raw/`
2. **复习数据获取**：综合运用了第一章介绍的 akshare 接口，可作为课后练习参考
3. **项目模板**：下载逻辑（检查已存在 → 下载 → 验证 → 记录日志）是真实项目中可直接复用的标准写法

**使用说明：** 第一次使用时从头运行本文件，之后每次运行会自动跳过已存在的文件，只补充缺失的部分。

| 数据集 | 存放路径 | 粒度 | 行数 |
|--------|----------|------|------|
| A 个股日度行情 | `data_raw/stock_daily/` | firm-date | ~15,100 |
| B1 市场指数日度 | `data_raw/index_daily/` | index-date | ~4,533 |
| B2 宏观月度 | `data_raw/macro_monthly/` | date | ~223 |
| C 年度财务指标 | `data_raw/fin_annual/` | firm-year | ~60 |
| D 公司基本信息 | `data_raw/company_info.csv` | firm | 10 |

In [ ]:
# ── 依赖库 ───────────────────────────────────────────────
import akshare as ak
import pandas as pd
import time
import os
from pathlib import Path
from datetime import datetime

# ── 目录结构 ──────────────────────────────────────────────
BASE = Path('data_raw')
dirs = [
    BASE / 'stock_daily',
    BASE / 'index_daily',
    BASE / 'macro_monthly',
    BASE / 'fin_annual',
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

# ── 下载日志 ──────────────────────────────────────────────
LOG_PATH = BASE / 'download_log.txt'

def log(msg: str, also_print: bool = True):
    """把日志同时写入文件和打印到屏幕"""
    ts  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{ts}] {msg}'
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')
    if also_print:
        print(line)

# 日志文件开头写入本次运行时间
with open(LOG_PATH, 'a', encoding='utf-8') as f:
    f.write(f'\n{"="*60}\n')
    f.write(f'运行时间：{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'{"="*60}\n')

print('目录和日志初始化完成')

## 数据集 A：10 只个股日度行情

覆盖金融、消费、制造、新能源、医药五个行业，兼顾大市值和中市值、国企和民企，方便后续做行业对比分析。

In [ ]:
# 股票列表：行业分散，有大有小，有国企有民企
STOCKS = {
    '000001': '平安银行',   # 银行
    '000002': '万科A',      # 房地产
    '600519': '贵州茅台',   # 白酒
    '300750': '宁德时代',   # 新能源
    '601318': '中国平安',   # 保险
    '002594': '比亚迪',     # 汽车
    '600036': '招商银行',   # 银行
    '601012': '隆基绿能',   # 光伏
    '000333': '美的集团',   # 家电
    '600276': '恒瑞医药',   # 医药
}

log('=' * 60)
log('开始下载数据集 A：个股日度行情')
log('=' * 60)

for code, name in STOCKS.items():
    save_path = BASE / 'stock_daily' / f'{code}.csv'

    # 已存在则跳过，避免重复下载
    if save_path.exists():
        n = len(pd.read_csv(save_path))
        log(f'✓ stock_daily/{code}.csv 已存在，跳过（{n} 行）')
        continue

    try:
        df = ak.stock_zh_a_hist(
            symbol     = code,
            period     = 'daily',
            start_date = '20200101',
            end_date   = '20261231',
            adjust     = 'hfq'          # 后复权，适合长期收益分析
        )

        # 统一列名为英文，方便后续 SQL 查询
        df.columns = ['date','open','close','high','low',
                      'volume','amount','amplitude','pct_chg','chg','turnover']
        df['code'] = code
        df['name'] = name

        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ stock_daily/{code}.csv  {len(df):,} 行  '
            f'{df["date"].min()} ~ {df["date"].max()}')

    except Exception as e:
        log(f'✗ {code} ({name}) 下载失败：{e}')

    time.sleep(0.8)   # 避免触发 akshare 限频

## 数据集 B1：市场指数日度

In [ ]:
INDICES = {
    '000300': '沪深300',
    '000001': '上证综指',
    '399001': '深证成指',
}

log('=' * 60)
log('开始下载数据集 B1：市场指数日度')
log('=' * 60)

for code, name in INDICES.items():
    save_path = BASE / 'index_daily' / f'{code}.csv'

    if save_path.exists():
        n = len(pd.read_csv(save_path))
        log(f'✓ index_daily/{code}.csv 已存在，跳过（{n} 行）')
        continue

    try:
        df = ak.index_zh_a_hist(
            symbol     = code,
            period     = 'daily',
            start_date = '20200101',
            end_date   = '20261231'
        )
        df.columns = ['date','open','close','high','low',
                      'volume','amount','amplitude','pct_chg','chg','turnover']
        df['index_code'] = code
        df['index_name'] = name

        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ index_daily/{code}.csv  {len(df):,} 行  '
            f'{df["date"].min()} ~ {df["date"].max()}')

    except Exception as e:
        log(f'✗ {code} ({name}) 下载失败：{e}')

    time.sleep(0.8)

## 数据集 B2：宏观月度数据

三个指标覆盖利率、汇率、通胀，月度频率，与个股日度数据形成「混频」结构，是后续 SQL JOIN 演示的核心素材。

In [ ]:
log('=' * 60)
log('开始下载数据集 B2：宏观月度数据')
log('=' * 60)

# ── Shibor 3 个月期 ──────────────────────────────────────
save_path = BASE / 'macro_monthly' / 'shibor_3m.csv'
if save_path.exists():
    log(f'✓ macro_monthly/shibor_3m.csv 已存在，跳过（{len(pd.read_csv(save_path))} 行）')
else:
    try:
        df = ak.rate_interbank(
            market    = '上海银行同业拆借市场',
            symbol    = 'Shibor人民币',
            indicator = '3月'
        )
        df.columns = ['date', 'shibor_3m']
        df['date'] = pd.to_datetime(df['date']).dt.to_period('M').astype(str)
        df = df[df['date'] >= '2020-01'].drop_duplicates('date')
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ shibor_3m.csv  {len(df)} 行  {df["date"].min()} ~ {df["date"].max()}')
    except Exception as e:
        log(f'✗ Shibor 下载失败：{e}')

time.sleep(0.8)

# ── 人民币兑美元汇率（月末值）────────────────────────────
save_path = BASE / 'macro_monthly' / 'usd_cny.csv'
if save_path.exists():
    log(f'✓ macro_monthly/usd_cny.csv 已存在，跳过（{len(pd.read_csv(save_path))} 行）')
else:
    try:
        df = ak.currency_hist(symbol='USDCNY')
        df = df[['日期', '收盘']].rename(columns={'日期': 'date', '收盘': 'usd_cny'})
        df['date'] = pd.to_datetime(df['date'])
        df = df[df['date'] >= '2020-01-01']
        # 按月取最后一个交易日，作为月末汇率
        df = df.resample('ME', on='date').last().reset_index()
        df['date'] = df['date'].dt.to_period('M').astype(str)
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ usd_cny.csv  {len(df)} 行  {df["date"].min()} ~ {df["date"].max()}')
    except Exception as e:
        log(f'✗ 汇率下载失败：{e}')

time.sleep(0.8)

# ── CPI 月度同比 ─────────────────────────────────────────
save_path = BASE / 'macro_monthly' / 'cpi_monthly.csv'
if save_path.exists():
    log(f'✓ macro_monthly/cpi_monthly.csv 已存在，跳过（{len(pd.read_csv(save_path))} 行）')
else:
    try:
        df = ak.macro_china_cpi_monthly()
        # 保留月份和全国当月同比两列
        df = df[['月份', '全国-当月']].rename(
            columns={'月份': 'date', '全国-当月': 'cpi_yoy'})
        df['date'] = df['date'].astype(str).str[:7]   # 取 YYYY-MM
        df = df[df['date'] >= '2020-01']
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ cpi_monthly.csv  {len(df)} 行  {df["date"].min()} ~ {df["date"].max()}')
    except Exception as e:
        log(f'✗ CPI 下载失败：{e}')

## 数据集 C：年度财务指标

In [ ]:
log('=' * 60)
log('开始下载数据集 C：年度财务指标')
log('=' * 60)

save_path = BASE / 'fin_annual' / 'fin_indicators.csv'

if save_path.exists():
    n = len(pd.read_csv(save_path))
    log(f'✓ fin_annual/fin_indicators.csv 已存在，跳过（{n} 行）')
else:
    all_fin = []

    for code, name in STOCKS.items():
        try:
            df = ak.stock_a_indicator_lg(symbol=code)
            df['code'] = code
            df['name'] = name
            # 保留分析常用字段
            keep = ['code', 'name', 'trade_date', 'pe_ttm', 'pb', 'ps_ttm', 'dv_ttm']
            df = df[[c for c in keep if c in df.columns]]
            # 提取年份，只保留年末数据（每年最后一个交易日）
            df['trade_date'] = pd.to_datetime(df['trade_date'])
            df['year'] = df['trade_date'].dt.year
            df = df[df['year'] >= 2020]
            df = df.sort_values('trade_date').groupby(['code', 'year']).last().reset_index()
            all_fin.append(df)
            log(f'  ✓ {code} ({name})：{len(df)} 年')

        except Exception as e:
            log(f'  ✗ {code} ({name}) 失败：{e}')

        time.sleep(0.8)

    if all_fin:
        df_fin = pd.concat(all_fin, ignore_index=True)
        df_fin.to_csv(save_path, index=False, encoding='utf-8-sig')
        log(f'✓ fin_indicators.csv 已保存  {len(df_fin)} 行')

## 数据集 D：公司基本信息

In [ ]:
log('=' * 60)
log('开始下载数据集 D：公司基本信息')
log('=' * 60)

save_path = BASE / 'company_info.csv'

if save_path.exists():
    log(f'✓ company_info.csv 已存在，跳过（{len(pd.read_csv(save_path))} 行）')
else:
    records = []

    for code, name in STOCKS.items():
        try:
            # akshare 返回的是 key-value 格式，需要转置为一行
            df = ak.stock_individual_info_em(symbol=code)
            info = dict(zip(df.iloc[:, 0], df.iloc[:, 1]))

            # 从实控人字段推断国企/民企
            controller = str(info.get('实控人', '') or info.get('控股股东', ''))
            ownership  = '国企' if any(kw in controller for kw in ['国资', '国有', '国家', '政府', '部']) else '民企'

            records.append({
                'code'       : code,
                'name'       : name,
                'industry_l1': info.get('行业', ''),
                'province'   : info.get('地区', ''),
                'list_date'  : info.get('上市时间', ''),
                'ownership'  : ownership,
            })
            log(f'  ✓ {code} {name}  {ownership}')

        except Exception as e:
            log(f'  ✗ {code} ({name}) 失败：{e}')
            records.append({'code': code, 'name': name})

        time.sleep(0.5)

    df_info = pd.DataFrame(records)
    df_info.to_csv(save_path, index=False, encoding='utf-8-sig')
    log(f'✓ company_info.csv 已保存  {len(df_info)} 行')

## 数据健康检查

所有下载完成后，运行这一节快速确认数据质量。

In [ ]:
import glob

print(f'{'数据集':<20} {'文件数':>6} {'总行数':>10} {'时间范围':<30} {'缺失值':>8}')
print('=' * 80)

# ── 个股日度 ──────────────────────────────────────────────
files = [f for f in glob.glob(str(BASE / 'stock_daily/*.csv'))]
if files:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'{'stock_daily':<20} {len(files):>6} {len(df):>10,} '
          f'{df["date"].min()} ~ {df["date"].max():<12} {df.isnull().sum().sum():>8}')

# ── 指数日度 ──────────────────────────────────────────────
files = glob.glob(str(BASE / 'index_daily/*.csv'))
if files:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'{'index_daily':<20} {len(files):>6} {len(df):>10,} '
          f'{df["date"].min()} ~ {df["date"].max():<12} {df.isnull().sum().sum():>8}')

# ── 宏观月度 ──────────────────────────────────────────────
files = glob.glob(str(BASE / 'macro_monthly/*.csv'))
if files:
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    all_dates = pd.read_csv(files[0])['date']
    print(f'{'macro_monthly':<20} {len(files):>6} {len(df):>10,} '
          f'{all_dates.min()} ~ {all_dates.max():<16} {df.isnull().sum().sum():>8}')

# ── 年度财务 ──────────────────────────────────────────────
p = BASE / 'fin_annual' / 'fin_indicators.csv'
if p.exists():
    df = pd.read_csv(p)
    print(f'{'fin_annual':<20} {'1':>6} {len(df):>10,} '
          f'{df["year"].min()} ~ {df["year"].max():<20} {df.isnull().sum().sum():>8}')

# ── 公司信息 ──────────────────────────────────────────────
p = BASE / 'company_info.csv'
if p.exists():
    df = pd.read_csv(p)
    print(f'{'company_info':<20} {'1':>6} {len(df):>10,} {"—":<30} {df.isnull().sum().sum():>8}')

print('=' * 80)
print('\n数据下载完成！可以打开 data_store_manage.ipynb 开始学习。')